In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# skip pipelines due to version compatibility issues
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()

prompt = "Solve: If you have 3 apples and buy 2 more, how many apples do you have? Answer:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# multi turn chat - from chatGPT
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = [
    {"role": "system", "content": "You are a helpful assistant. Always reply in English only."}
]

def build_prompt(chat_history):
    messages = []
    for turn in chat_history:
        messages.append({
            "role": turn["role"],
            "content": turn["content"]
        })
    return tokenizer.apply_chat_template(
        messages, # tuple-role, content
        tokenize=False,
        add_generation_prompt=True
    )

print("Hello! How are you? (type 'exit' or 'quit' to end the conversation)")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    print("User Input: ", user_input)
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7
    )
    
    # get only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


In [ ]:
# edited by me
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = []

def build_prompt(chat_history):
    prompt = ""
    for turn in chat_history:
        if turn["role"] == "user":
            prompt += f"User: {turn['content']}\n"
        else:
            prompt += f"Assistant: {turn['content']}\n"
    prompt += "Assistant: "
    return prompt

print("Hello! Welcome to chatLLM")
while True:
    print("Please provide an input ")
    user_input = input("You: ")
    print(f"User Input: {user_input}")
    if user_input.lower() in ["exit"]:
        break
    
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract only new response
    assistant_reply = response[len(prompt):].strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


In [ ]:
# GSM-8K evaluator benchmark - chatgpt
# load GSM8K dataset
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")
test_data = dataset["test"]

print(test_data[0])

In [ ]:
# load model
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
)

# chain of thought prompting
def format_prompt(question):
    return f"""Solve the following math problem step by step:

Question: {question}
Answer:"""

In [ ]:
# run inference on 1 example
example = test_data[0]

prompt = format_prompt(example["question"])

output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(output[0]["generated_text"])

In [ ]:
# extract final answer
import re

def extract_answer(text):
    match = re.search(r"#### (\d+)", text)
    return match.group(1) if match else None
print(extract_answer(output[0]["generated_text"]))

In [ ]:
from deepeval.benchmarks import GSM8K

class HFModel:
    def __init__(self, generator):
        self.generator = generator

    def generate(self, prompt: str) -> str:
        output = self.generator(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )
        return output[0]["generated_text"]
hf_model = HFModel(generator)
print(hf_model.generate("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"))
benchmark = GSM8K(
    n_problems=10,
    n_shots=3,
    enable_cot=True
)

benchmark.evaluate(model=hf_model)
print(benchmark.overall_score)

In [ ]:
# CURRENT-USE THIS

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# skip pipelines due to version compatibility issues
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems.

Q: Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally?
A: The total cost is (5 * $3) + (4 * $1.20) + (5 * $0.5) + $2.7 = $15 + $4.8 + $2.5 + $2.7 = $25. A: 5

Q: A shop has 10 candies. It sells 4. How many are left?
A: The shop had 10 candies and sold 4. 10 - 4 = 6. Answer: 6

Q: {question}
A:"""

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    full_text = output[0]["generated_text"]

    # remove prompt
    answer = full_text[len(prompt):].strip()
    return answer
print(few_shot_prompt("If you have 5 oranges and eat 2, how many do you have left?"))

prompt = "Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally?"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally? To determine how much each friend will pay, we first need to calculate the total cost of all the items ordered by the five friends.

The costs are as follows:
- 5 hamburgers at $3 each: \(5 \times 3 = 15\) dollars
- 4 sets of French fries at $1.20 each: \(4 \times 1.20 = 4.80\) dollars
- 5 cups of soda at $0.5 each: \(5 \times 0.5 = 2.50\) dollars
- 1 platter of spaghetti at $2.70

Next, we sum these amounts to find the total bill:

\[
15 + 4.80 + 2.50 + 2.70 = 24.00 \text{ dollars}
\]

Since there are 5 friends splitting the bill equally, we divide the total bill by 5:

\[
\frac{24.00}{5} = 4.80 \text{ dollars per person}
\]

Thus, each friend will pay \(\boxed{4.80}\) dollars.


In [1]:
# zero shot prompting
from transformers import pipeline

# Use an instruction-tuned model
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct"
)

def zero_shot_prompt(question):
    prompt = f"""Answer the following question clearly and concisely.

Question: {question}
Answer:"""

    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )
  # Remove prompt from output
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):].strip()

    return answer

zero_shot_prompt("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?")

W0401 10:44:55.901000 17084 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


''

In [ ]:
# few shot prompting
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",   # replace later if needed
    device=-1
)

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems.

Q: If you have 3 apples and buy 2 more, how many apples do you have?
A: You start with 3 apples and buy 2 more. 3 + 2 = 5. Answer: 5

Q: A shop has 10 candies. It sells 4. How many are left?
A: The shop had 10 candies and sold 4. 10 - 4 = 6. Answer: 6

Q: {question}
A:"""

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    full_text = output[0]["generated_text"]

    # remove prompt
    answer = full_text[len(prompt):].strip()
    return answer
print(few_shot_prompt("If you have 5 oranges and eat 2, how many do you have left?"))